# Lightweight Processing + Visualization

This notebook demonstrates the lightweight streaming flow with a **declarative transform spec**.
You define transform rules, and the library executes them before analysis.


In [ ]:
import sys
from datetime import date, timedelta
from pathlib import Path


def _find_project_src(start: Path) -> Path:
    # Prefer walking up from the current working directory.
    for base in [start, *start.parents]:
        candidate = base / "src" / "data_ingestion"
        if candidate.exists():
            return base / "src"

    # Explicit fallback: two directories up from cwd.
    two_up = start.parent.parent
    candidate = two_up / "src" / "data_ingestion"
    if candidate.exists():
        return two_up / "src"

    raise RuntimeError("Could not locate project src/data_ingestion from notebook cwd")


SRC_PATH = _find_project_src(Path.cwd().resolve())
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib.pyplot as plt
import pandas as pd

from data_ingestion.pipeline import stream_transformed_records

plt.style.use("ggplot")

In [ ]:
# Keep date range small to minimize API usage.
end_date = date.today()
start_date = end_date - timedelta(days=7)

fetcher_specs = [
    {
        "source": "openalex",
        "config": {
            "query": "data engineering",
            "max_pages": 1,
            "per_page": 50,
        },
    },
    {
        "source": "crossref",
        "config": {
            "query": "data engineering",
            "max_pages": 1,
            "rows": 50,
        },
    },
]

transform_spec = {
    "transforms": [
        {"op": "require_fields", "fields": ["title", "url"]},
        {
            "op": "include_terms",
            "terms": ["data", "engineering", "pipeline", "etl"],
            "fields": ["title", "abstract", "topic"],
        },
        {"op": "dedupe", "keys": ["source", "external_id", "url"]},
    ]
}

rows = []
for source, record in stream_transformed_records(
    fetcher_specs,
    transform_spec=transform_spec,
    start_date=start_date.isoformat(),
    end_date=end_date.isoformat(),
):
    rows.append(
        {
            "source": source,
            "external_id": record.external_id,
            "title": record.title,
            "published_date": (
                record.published_date.isoformat()
                if record.published_date is not None
                else None
            ),
            "topic": record.topic,
        }
    )

df = pd.DataFrame(rows)
df.head()

In [ ]:
print(f"Total rows: {len(df)}")
df["source"].value_counts()

In [ ]:
# Visualization 1: records by source
ax = (
    df["source"]
    .value_counts()
    .plot(kind="bar", figsize=(7, 4), title="Records by Source (Lightweight)")
)
ax.set_xlabel("Source")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: top topics from declarative transformation
top_topics = df["topic"].fillna("unknown").value_counts().head(10)
ax = top_topics.plot(kind="barh", figsize=(8, 5), title="Top Topics (Lightweight)")
ax.set_xlabel("Count")
ax.set_ylabel("Topic")
plt.tight_layout()
plt.show()

## Expanded Analysis

These additional checks help validate data quality, understand temporal behavior, and inspect term-level patterns in transformed outputs.


In [ ]:
# Analysis 1: data quality snapshot
analysis_df = df.copy()
analysis_df["published_date"] = pd.to_datetime(
    analysis_df["published_date"], errors="coerce"
)

quality = pd.DataFrame(
    {
        "rows": [len(analysis_df)],
        "unique_sources": [analysis_df["source"].nunique(dropna=True)],
        "unique_external_ids": [analysis_df["external_id"].nunique(dropna=True)],
        "missing_title_pct": [analysis_df["title"].isna().mean() * 100],
        "missing_topic_pct": [analysis_df["topic"].isna().mean() * 100],
        "missing_published_date_pct": [
            analysis_df["published_date"].isna().mean() * 100
        ],
    }
)
quality

In [ ]:
# Analysis 2: daily volume by source
daily_by_source = (
    analysis_df.dropna(subset=["published_date"])
    .groupby([analysis_df["published_date"].dt.date, "source"])
    .size()
    .unstack(fill_value=0)
)

if daily_by_source.empty:
    print("No valid published_date values available for daily trend analysis.")
else:
    ax = daily_by_source.plot(
        kind="line",
        marker="o",
        figsize=(9, 4),
        title="Daily Volume by Source (Lightweight)",
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("Records")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Analysis 3: source-topic matrix (top topics)
top_topic_labels = analysis_df["topic"].fillna("unknown").value_counts().head(8).index
matrix = pd.crosstab(
    analysis_df["source"],
    analysis_df["topic"].fillna("unknown"),
)[top_topic_labels]

if matrix.empty:
    print("No source/topic data available for matrix analysis.")
else:
    ax = matrix.plot(
        kind="bar",
        stacked=True,
        figsize=(10, 5),
        title="Top Topic Mix by Source (Lightweight)",
    )
    ax.set_xlabel("Source")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()

In [ ]:
# Analysis 4: top terms in titles
stop_words = {
    "the",
    "and",
    "for",
    "with",
    "from",
    "into",
    "data",
    "engineering",
    "using",
    "study",
    "system",
    "analysis",
    "model",
    "models",
}
title_tokens = (
    analysis_df["title"].fillna("").str.lower().str.findall(r"[a-z]{3,}").explode()
)
title_tokens = title_tokens[title_tokens.notna() & ~title_tokens.isin(stop_words)]
top_terms = title_tokens.value_counts().head(15)

if top_terms.empty:
    print("No title token data available for term-frequency analysis.")
else:
    ax = top_terms.sort_values().plot(
        kind="barh",
        figsize=(8, 5),
        title="Top Title Terms (Lightweight)",
    )
    ax.set_xlabel("Count")
    ax.set_ylabel("Term")
    plt.tight_layout()
    plt.show()